# Tutorial: SCARLET workflow from raw data

This notebook presents a complete, tutorial-style workflow to:
- initialize a `WorkflowContext` from a raw-data directory,
- inspect detected runs and configurations,
- generate the `refs_sub` and `refs_norm` reference files,
- correct the normalization references,
- compute transmissions,
- produce the final `I(q)` text files.

The idea is to keep a simple sequence throughout the notebook: **load, inspect, correct, reference, integrate**.


## 1. Environment setup

This first cell moves the notebook to the repository root and adds `src/` to the `PYTHONPATH`.
This is useful whether the notebook is launched from `notebooks/` or from the repository root.


In [ ]:
from pathlib import Path
import os
import sys

# Detect the repository root and make sure src/ is importable.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repository root: {ROOT}")
print(f"Python path includes: {SRC}")


## 2. Define the working paths

Here we define:
- the raw-data directory,
- the output directory,
- the instrument name used for conversion and file interpretation.


In [ ]:
from scarlet.workflow.context import initialize_workflow_context_from_raw_directory

# Adapt these paths to your own experiment if needed.
RAW_DIR = ROOT / "data" / "SANSLLB" / "raw"
OUTPUT_DIR = ROOT / "data" / "SANSLLB" / "out"
INSTRUMENT_NAME = "sansllb"

RAW_DIR, OUTPUT_DIR


## 3. Initialize the `WorkflowContext`

This step scans the raw files, converts what needs to be converted, detects runs,
builds the instrumental configurations, and stores everything in a `WorkflowContext` object.


In [ ]:
# Build the workflow context from the raw-data directory.
w = initialize_workflow_context_from_raw_directory(
    RAW_DIR,
    instrument_name=INSTRUMENT_NAME,
    output_dir=OUTPUT_DIR,
    overwrite=True,
)

print(f"runs: {len(w.runs)}")
print(f"configurations: {len(w.configurations)}")
print(f"issues: {len(w.issues)}")
print(f"artifacts: {len(w.artifacts)}")


## 4. Inspect the detected runs

The table below is useful to verify that each file has been classified with:
- a `config_id`,
- a `mode` (`scattering` or `transmission`),
- an `entity` (`sample`, `empty_cell`, `dark`, etc.).

This is a good checkpoint before generating the reference files.


In [4]:
w.runs_table()

sample_name,config_id,mode,entity,file_path
empty_beam_att5_ws_beamstop,config_1,transmission,empty_beam,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002339.nxs
empty_cell,config_1,transmission,empty_cell,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002341.nxs
H2O,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002342.nxs
S1_P_PB_25_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002343.nxs
S2_P_PB_60_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002344.nxs
S3_P_PBK_40_2_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002345.nxs
S4_P_BC_25_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002346.nxs
S5_P_BC_60_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002347.nxs
S6_P_BCK_40_2_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002348.nxs
S7_P_MI_25_1mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002349.nxs


## 5. Inspect the configurations

This table summarizes the instrumental properties detected for each configuration:
wavelength, sample-to-detector distance, collimation, apertures, and related metadata.


In [5]:
w.configurations_table()

config_id,wavelength,sample_detector_distance,notes,has_collimation,collimation_distance,last_aperture_to_sample_distance,aperture1_type,aperture1_x_gap,aperture1_y_gap,aperture1_diameter,aperture2_type,aperture2_x_gap,aperture2_y_gap,aperture2_diameter
config_1,6.00046,2.5,,True,0.5,1.2,slit,0.0419996,0.0419996,,slit,0.0319997,0.0320003,
config_2,5.9989,9,,True,2.58333,1.2,slit,0.0269997,0.0269999,,slit,0.0169996,0.0169998,


## 6. Generate the reference files

At this stage we create, for each configuration:
- one `refs_sub` file for subtraction,
- one `refs_norm` file for normalization.

These files gather the experimental references into a single format that is easier to reuse later.


In [ ]:
from scarlet.workflow.context import generate_reference_files_from_workflow_context

# Generate refs_sub and refs_norm for every detected configuration.
generate_reference_files_from_workflow_context(w)
print("refs_sub:", {k: str(v) for k, v in sorted(w.refs_sub_files.items())})
print("refs_norm:", {k: str(v) for k, v in sorted(w.refs_norm_files.items())})


## 7. Update the beam center for `detector0`

The transmission detector beam center can be re-estimated from the
`empty_beam_transmission` image by using the center of mass of the transmission ROI.

This step updates the `refs_sub` files that were just generated.


In [ ]:
from scarlet.workflow import update_detector0_beam_center_from_empty_beam_transmission

# Update detector0 beam center in each refs_sub from the empty-beam transmission frame.
for config_id, refs_sub_path in sorted(w.refs_sub_files.items()):
    update_detector0_beam_center_from_empty_beam_transmission(refs_sub_path)
    print(config_id, refs_sub_path)


## 8. Optional manual mask editing

If needed, the graphical editor can be used to draw detector masks.
The cell remains commented out to avoid opening the GUI by mistake.


In [ ]:
from scarlet.gui import run_mask_editor

# Uncomment to launch the GUI mask editor on a given raw NeXus file.
# run_mask_editor()


## 9. Export the runs table to CSV

The `runs_report.csv` file is useful if you want to manually adjust:
- sample names,
- `entity` values,
- `mode` values,
- or remove rows that should not remain in the workflow.


In [ ]:
from scarlet.workflow.context import write_runs_report_csv


In [ ]:
# Example: reload the workflow after manually editing the CSV file.
# from scarlet.workflow.context import update_workflow_context_from_runs_report_csv
# update_workflow_context_from_runs_report_csv(w, "data/SANSLLB/out/runs_report_modified.csv")
# update_workflow_context_from_runs_report_csv(w, "data/SANSLLB/out/runs_report.csv")
# file = write_runs_report_csv(w, csv_path="data/SANSLLB/out/runs_report_updated.csv", overwrite=True)


In [ ]:
# Example: reload the workflow after manually editing the CSV.
# from scarlet.workflow.context import update_workflow_context_from_runs_report_csv
# update_workflow_context_from_runs_report_csv(w, "data/SANSLLB/out/runs_report_modified.csv")
# update_workflow_context_from_runs_report_csv(w, "data/SANSLLB/out/runs_report.csv")
# file = write_runs_report_csv(w, csv_path="data/SANSLLB/out/runs_report_updated.csv", overwrite=True)


## 10. Compute reference transmissions

Here we compute the scalar transmissions for the references embedded in the
`refs_sub` and `refs_norm` bundles, using the stored transmission ROI.


In [ ]:
from scarlet.reduction.transmission import compute_reference_transmissions

# Compute and store reference transmissions inside refs_norm and refs_sub bundles.
for config_id, path in sorted(w.refs_norm_files.items()):
    compute_reference_transmissions(path)
    print("refs_norm", config_id, path)
for config_id, path in sorted(w.refs_sub_files.items()):
    compute_reference_transmissions(path)
    print("refs_sub", config_id, path)


## 11. Update masks in the reference files

If SCARLET mask files have been generated in `out/`, this step finds them and
injects them back into the matching `refs_sub` and `refs_norm` files.


In [ ]:
from scarlet.workflow import update_reference_masks_from_workflow_context

# Synchronize mask files found in the output directory back into refs_sub / refs_norm.
w = update_reference_masks_from_workflow_context(w)


## 12. Build `water_corrected` in the `refs_norm` files

This step prepares the final normalization reference from the water references,
including the required subtractions and the solid-angle normalization.


In [ ]:
from scarlet.workflow import write_corrected_water_scattering

# Build and store the water_corrected reference in every refs_norm file.
for config_id, refs_norm_path in sorted(w.refs_norm_files.items()):
    write_corrected_water_scattering(refs_norm_path)
    print(config_id, refs_norm_path)


## 13. Save and reload the workflow context

The `WorkflowContext` can be serialized into a dedicated NeXus file, which makes it
possible to resume the processing later without starting again from scratch.


In [ ]:
from scarlet.workflow import save_workflow_context, load_workflow_context

# Persist the workflow state and reload it to validate the round-trip.
save_workflow_context(w, "workflow_context.nxs")
w2 = load_workflow_context("workflow_context.nxs")


## 14. Compute sample transmissions

Using the sample transmission files together with the `refs_sub` bundles,
this step fills `w.transmissions` for the scattering treatment.


In [ ]:
from scarlet.workflow.context import update_transmissions_from_workflow_context

# Compute sample transmissions for all configurations in the workflow.
update_transmissions_from_workflow_context(w)
w.transmissions_table()


## 15. Final integration of the scattering files

This is the final tutorial step:
- monitor normalization,
- reference subtraction,
- solid-angle normalization,
- division by `water_corrected`,
- azimuthal averaging,
- writing the final `I(q)` text files.

Each detector produces its own output text file inside `w.output_dir`.


In [ ]:
from scarlet.workflow import integrate_scattering_from_workflow_context

# Produce one I(q) text file per detector and per sample/configuration.
integrate_scattering_from_workflow_context(w, n_bins=200)
